In [ ]:
from IPython.display import HTML, display
display(HTML('''
<div style="text-align:right;padding:8px 0;">
<button id="toggle-code-btn" style="
  padding:8px 18px;font-size:14px;cursor:pointer;
  background:#4a90d9;color:white;border:none;border-radius:5px;
  box-shadow:0 2px 4px rgba(0,0,0,0.2);
" onclick="
  var cells = document.querySelectorAll('.jp-Cell-inputWrapper');
  var btn = document.getElementById('toggle-code-btn');
  var show = btn.dataset.show !== 'true';
  btn.dataset.show = show;
  btn.textContent = show ? '隐藏代码 [Hide]' : '显示代码 [Show]';
  cells.forEach(function(c){ c.style.display = show ? '' : 'none'; });
">显示代码 [Show]</button>
</div>
'''))


# 第4章 处理器基础 - 4.3 位操作 (Bit Manipulation)

## Cambridge International AS & A Level Computer Science (9618)

---

> **核心问题**：计算机如何在最底层——单个比特(bit)的层面——操纵数据？

位操作是计算机科学中最基础、最强大的技术之一。掌握它，你就真正理解了计算机的"语言"。

## 1. 二进制移位 (Binary Shifts)

移位操作就是将二进制数的所有位向左或向右移动。

> **为什么移位很重要？**
> - 左移1位 = 乘以2（比乘法运算快得多！）
> - 右移1位 = 除以2
> - CPU执行移位操作只需要1个时钟周期，而乘法可能需要多个周期
> - 在嵌入式系统和性能关键的代码中大量使用

### 移位类型总览

| 类型 | 方向 | 空位填充 | 移出的位 | 用途 |
|------|------|---------|---------|------|
| 逻辑左移 | ← | 右边填0 | 丢弃 | 无符号乘2 |
| 逻辑右移 | → | 左边填0 | 丢弃 | 无符号除2 |
| 算术左移 | ← | 右边填0 | 丢弃(保留符号位) | 有符号乘2 |
| 算术右移 | → | 左边填符号位 | 丢弃 | 有符号除2 |
| 循环移位 | ←/→ | 移出的位填入另一端 | 循环 | 加密、校验 |

## 2. 逻辑移位 (Logical Shift)

### 2.1 逻辑左移 (Logical Left Shift)

In [ ]:
%matplotlib inline

# === 中文字体配置 (Chinese Font Setup) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

# 方法1: 全局设置 WenQuanYi Zen Hei 字体
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 方法2: 强制重建字体缓存（首次运行可能需要）
fm._load_fontmanager(try_read_cache=False)

# 验证
test_font = fm.findfont('WenQuanYi Zen Hei')
if 'WenQuanYi Zen Hei' in test_font or 'wqy' in test_font.lower():
    print('✅ 中文字体 WenQuanYi Zen Hei 已加载')
else:
    print(f'⚠️ WenQuanYi Zen Hei 未找到，使用: {test_font}')
    # Fallback: 直接注册字体文件
    font_path = '/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc'
    if __import__('os').path.exists(font_path):
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei'] + plt.rcParams['font.sans-serif']
        print('✅ 已手动加载 WenQuanYi Zen Hei 字体文件')

import matplotlib.patches as mpatches

def visualize_shift(original, shifted, title, shift_type, fig_ax=None):
    if fig_ax:
        fig, ax = fig_ax
    else:
        fig, ax = plt.subplots(figsize=(12, 4))
    fig.patch.set_facecolor('#1a1a2e')
    ax.set_facecolor('#1a1a2e')
    ax.axis('off')
    ax.set_xlim(-0.5, 12)
    ax.set_ylim(-0.5, 4)
    
    # Draw original bits
    ax.text(0, 3.3, 'Before:', fontsize=10, color='white', fontweight='bold')
    for i, bit in enumerate(original):
        x = 2 + i * 1.1
        box = mpatches.FancyBboxPatch((x, 2.8), 0.9, 0.8,
            boxstyle='round,pad=0.05', facecolor='#0f3460',
            edgecolor='#00d2ff', linewidth=1.5)
        ax.add_patch(box)
        ax.text(x+0.45, 3.2, str(bit), fontsize=12, fontweight='bold',
            color='#00d2ff', ha='center')
    
    # Draw shifted bits
    ax.text(0, 1.3, 'After:', fontsize=10, color='white', fontweight='bold')
    for i, bit in enumerate(shifted):
        x = 2 + i * 1.1
        is_new = False
        if shift_type == 'left' and i >= len(shifted) - 1:
            is_new = True
        elif shift_type == 'right' and i == 0:
            is_new = True
        ec = '#e94560' if is_new else '#53d769'
        fc = '#2d1f1f' if is_new else '#0f3460'
        box = mpatches.FancyBboxPatch((x, 0.8), 0.9, 0.8,
            boxstyle='round,pad=0.05', facecolor=fc,
            edgecolor=ec, linewidth=1.5)
        ax.add_patch(box)
        ax.text(x+0.45, 1.2, str(bit), fontsize=12, fontweight='bold',
            color=ec, ha='center')
    
    # Arrow
    direction = '\u2190 LEFT' if shift_type == 'left' else 'RIGHT \u2192'
    ax.text(6, 2.2, f'Shift {direction}', fontsize=11, color='#ffd700',
        ha='center', fontweight='bold')
    ax.set_title(title, fontsize=12, fontweight='bold', color='white', pad=5)
    return fig, ax

# Logical Left Shift: 00101100 (44) << 1 = 01011000 (88)
original = [0, 0, 1, 0, 1, 1, 0, 0]
shifted =  [0, 1, 0, 1, 1, 0, 0, 0]
fig, ax = visualize_shift(original, shifted,
    'Logical Left Shift: 00101100 (44) << 1 = 01011000 (88)', 'left')
ax.text(6, 0.2, '44 \u00d7 2 = 88 \u2714', fontsize=11, color='#53d769',
    ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

# Logical Right Shift
original2 = [0, 0, 1, 0, 1, 1, 0, 0]
shifted2  = [0, 0, 0, 1, 0, 1, 1, 0]
fig2, ax2 = visualize_shift(original2, shifted2,
    'Logical Right Shift: 00101100 (44) >> 1 = 00010110 (22)', 'right')
ax2.text(6, 0.2, '44 \u00f7 2 = 22 \u2714', fontsize=11, color='#53d769',
    ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


## 3. 算术移位 (Arithmetic Shift)

算术移位与逻辑移位的关键区别：**保留符号位**。

在二进制补码表示法中，最高位（最左边）是符号位：
- 0 = 正数
- 1 = 负数

### 算术右移
- 右移时，左边用 **符号位** 填充（不是用0）
- 这样负数右移后仍然是负数

```
11110100 (-12) 算术右移1位:
→ 11111010 (-6)  ✓ 保留了负号，相当于÷2

11110100 (-12) 逻辑右移1位:
→ 01111010 (+122)  ✗ 符号改变了！错误的结果！
```

In [ ]:
def show_arithmetic_shift():
    print('=== \u7b97\u672f\u79fb\u4f4d vs \u903b\u8f91\u79fb\u4f4d ===\n')
    
    def to_signed_8bit(n):
        if n < 0:
            return 256 + n
        return n
    
    def from_signed_8bit(bits):
        if bits >= 128:
            return bits - 256
        return bits
    
    values = [44, -12, 100, -100, 7, -7]
    
    print(f'{"\u503c":>6} | {"\u4e8c\u8fdb\u5236":>10} | {"\u7b97\u672f\u53f3\u79fb1":>12} | {"\u903b\u8f91\u53f3\u79fb1":>12} | {"\u7b97\u672f\u5de6\u79fb1":>12}')
    print('-' * 65)
    for val in values:
        bits = to_signed_8bit(val)
        binary = format(bits, '08b')
        
        # Arithmetic right shift
        sign_bit = binary[0]
        arith_right = sign_bit + binary[:-1]
        arith_right_val = from_signed_8bit(int(arith_right, 2))
        
        # Logical right shift
        logic_right = '0' + binary[:-1]
        logic_right_val = from_signed_8bit(int(logic_right, 2))
        
        # Arithmetic left shift (same as logical for left)
        arith_left = binary[1:] + '0'
        arith_left_val = from_signed_8bit(int(arith_left, 2))
        
        print(f'{val:>6} | {binary:>10} | {arith_right} ({arith_right_val:>4}) | {logic_right} ({logic_right_val:>4}) | {arith_left} ({arith_left_val:>4})')

show_arithmetic_shift()

## 4. 循环移位 (Cyclic Shift)

循环移位中，移出的位不会丢失，而是从另一端"重新进入"。

```
循环左移: 10110011 → 01100111 (最左边的1移到最右边)
循环右移: 10110011 → 11011001 (最右边的1移到最左边)
```

> **应用场景**：
> - 加密算法（如DES、AES使用循环移位）
> - 校验计算
> - 位图操作

In [ ]:
def cyclic_shift_demo():
    print('=== \u5faa\u73af\u79fb\u4f4d\u6f14\u793a ===\n')
    
    def cyclic_left(bits_str, n=1):
        for _ in range(n):
            bits_str = bits_str[1:] + bits_str[0]
        return bits_str
    
    def cyclic_right(bits_str, n=1):
        for _ in range(n):
            bits_str = bits_str[-1] + bits_str[:-1]
        return bits_str
    
    original = '10110011'
    print(f'\u539f\u59cb\u503c: {original} ({int(original, 2)})')
    print()
    
    print('\u5faa\u73af\u5de6\u79fb\uff1a')
    current = original
    for i in range(8):
        new = cyclic_left(current)
        moved = current[0]
        print(f'  Step {i+1}: {current} -> {new}  (bit \'{moved}\' \u4ece\u5de6\u79fb\u5230\u53f3)')
        current = new
    print(f'  8\u6b21\u5faa\u73af\u540e\u56de\u5230\u539f\u59cb\u503c: {current == original}')
    
    print()
    print('\u5faa\u73af\u53f3\u79fb\uff1a')
    current = original
    for i in range(4):
        new = cyclic_right(current)
        moved = current[-1]
        print(f'  Step {i+1}: {current} -> {new}  (bit \'{moved}\' \u4ece\u53f3\u79fb\u5230\u5de6)')
        current = new

cyclic_shift_demo()

## 5. 移位运算的数学意义

让我们用代码验证移位与乘除的关系：

In [ ]:
print('=== \u79fb\u4f4d = \u5feb\u901f\u4e58\u9664 ===\n')

value = 13
print(f'\u539f\u59cb\u503c: {value} = {value:08b}')
print()

for shift in range(1, 5):
    left = value << shift
    right = value >> shift
    multiplier = 2 ** shift
    print(f'\u5de6\u79fb{shift}\u4f4d: {value:08b} << {shift} = {left:08b} = {left:>4}  (= {value} \u00d7 {multiplier})')
    print(f'\u53f3\u79fb{shift}\u4f4d: {value:08b} >> {shift} = {right:08b} = {right:>4}  (= {value} \u00f7 {multiplier} = {value/multiplier:.1f})')
    print()

print('\u6ce8\u610f: \u53f3\u79fb\u7684\u9664\u6cd5\u662f\u6574\u6570\u9664\u6cd5(\u5411\u4e0b\u53d6\u6574)!')
print(f'\u4f8b\u5982: 13 >> 1 = {13 >> 1}, \u4f46 13/2 = 6.5')

## 6. 位掩码 (Bit Masking)

位掩码是使用逻辑运算来操纵特定位的技术。

### 6.1 AND 掩码 — 测试/清除位

**AND 的规则**: 只有两个都是1，结果才是1

```
  数据:   10110101
  掩码: & 00001111
  结果:   00000101  ← 只保留了低4位，高4位被"清除"了
```

**用途**：
- 测试某一位是否为1
- 清除（设为0）不需要的位
- 提取数据的一部分

### 6.2 OR 掩码 — 设置位

**OR 的规则**: 只要有一个是1，结果就是1

```
  数据:   10100000
  掩码: | 00001111
  结果:   10101111  ← 低4位被"设置"为1，高4位不变
```

**用途**：
- 将特定位设置为1
- 打开某些功能标志

### 6.3 XOR 掩码 — 翻转位

**XOR 的规则**: 两个不同时结果为1

```
  数据:   10110101
  掩码: ^ 11111111
  结果:   01001010  ← 所有位都被翻转了 (NOT)
```

**用途**：
- 翻转（切换）特定位
- 简单加密/解密（XOR两次恢复原值）
- 交换两个变量的值（不需要临时变量）

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(3, 1, figsize=(12, 9))
fig.patch.set_facecolor('#1a1a2e')

operations = [
    ('AND Mask (\u6d4b\u8bd5/\u6e05\u9664\u4f4d)', 0b10110101, 0b00001111, '&', '#e94560'),
    ('OR Mask (\u8bbe\u7f6e\u4f4d)', 0b10100000, 0b00001111, '|', '#53d769'),
    ('XOR Mask (\u7ffb\u8f6c\u4f4d)', 0b10110101, 0b11111111, '^', '#00d2ff'),
]

for idx, (title, data, mask, op, color) in enumerate(operations):
    ax = axes[idx]
    ax.set_facecolor('#16213e')
    ax.axis('off')
    ax.set_xlim(0, 14)
    ax.set_ylim(0, 4)
    
    result = eval(f'{data} {op} {mask}')
    data_bits = format(data, '08b')
    mask_bits = format(mask, '08b')
    result_bits = format(result, '08b')
    
    ax.text(0.5, 3.2, f'{title}', fontsize=11, fontweight='bold', color=color)
    
    labels = ['Data:', f'Mask ({op}):', 'Result:']
    values = [data_bits, mask_bits, result_bits]
    y_positions = [2.8, 1.8, 0.6]
    
    for row, (label, bits, y) in enumerate(zip(labels, values, y_positions)):
        ax.text(1.5, y, label, fontsize=9, color='white', ha='right')
        for j, bit in enumerate(bits):
            x = 2 + j * 1.2
            is_result = (row == 2)
            ec = color if is_result else '#555'
            lw_v = 2 if is_result else 1
            box = mpatches.FancyBboxPatch((x, y-0.25), 0.9, 0.6,
                boxstyle='round,pad=0.05', facecolor='#0f3460',
                edgecolor=ec, linewidth=lw_v)
            ax.add_patch(box)
            ax.text(x+0.45, y+0.05, bit, fontsize=12, fontweight='bold',
                color=color if is_result else 'white', ha='center')
        
        decimal_val = int(bits, 2)
        ax.text(12, y, f'= {decimal_val}', fontsize=10, color='white')
    
    # Separator line
    ax.plot([2, 11.5], [1.35, 1.35], color=color, lw=1.5, ls='--')

plt.suptitle('Bit Masking Operations', fontsize=14, fontweight='bold', color='white')
plt.tight_layout()
plt.show()

## 7. 真实世界应用：设备控制寄存器

在嵌入式系统中，硬件设备通常通过一个 **控制寄存器** 来管理。
寄存器的每一位控制一个功能：

### 示例：LED控制寄存器

```
Bit 7: 电源开关     (1=开, 0=关)
Bit 6: 亮度高位
Bit 5: 亮度低位
Bit 4: 闪烁模式     (1=闪烁, 0=常亮)
Bit 3: LED 红色     (1=开, 0=关)
Bit 2: LED 绿色     (1=开, 0=关)
Bit 1: LED 蓝色     (1=开, 0=关)
Bit 0: 未使用
```

In [ ]:
class DeviceRegister:
    BIT_NAMES = {
        7: '\u7535\u6e90', 6: '\u4eae\u5ea6H', 5: '\u4eae\u5ea6L', 4: '\u95ea\u70c1',
        3: '\u7ea2\u8272LED', 2: '\u7eff\u8272LED', 1: '\u84dd\u8272LED', 0: '\u672a\u4f7f\u7528'
    }
    
    def __init__(self, value=0):
        self.value = value & 0xFF
    
    def show(self):
        bits = format(self.value, '08b')
        print(f'\u5bc4\u5b58\u5668\u503c: {bits} (0x{self.value:02X})')
        for i in range(7, -1, -1):
            bit_val = (self.value >> i) & 1
            status = '\u5f00' if bit_val else '\u5173'
            print(f'  Bit {i}: {self.BIT_NAMES[i]:>8} = {bit_val} ({status})')
        print()
    
    def set_bit(self, bit_pos):
        mask = 1 << bit_pos
        print(f'\u8bbe\u7f6eBit {bit_pos} ({self.BIT_NAMES[bit_pos]}): OR\u63a9\u7801 = {format(mask, "08b")}')
        self.value = self.value | mask
    
    def clear_bit(self, bit_pos):
        mask = ~(1 << bit_pos) & 0xFF
        print(f'\u6e05\u9664Bit {bit_pos} ({self.BIT_NAMES[bit_pos]}): AND\u63a9\u7801 = {format(mask, "08b")}')
        self.value = self.value & mask
    
    def toggle_bit(self, bit_pos):
        mask = 1 << bit_pos
        print(f'\u7ffb\u8f6cBit {bit_pos} ({self.BIT_NAMES[bit_pos]}): XOR\u63a9\u7801 = {format(mask, "08b")}')
        self.value = self.value ^ mask
    
    def test_bit(self, bit_pos):
        mask = 1 << bit_pos
        result = (self.value & mask) != 0
        print(f'\u6d4b\u8bd5Bit {bit_pos} ({self.BIT_NAMES[bit_pos]}): {"\u5f00" if result else "\u5173"}')
        return result

# Demo
print('=== LED\u63a7\u5236\u5bc4\u5b58\u5668\u6f14\u793a ===\n')
reg = DeviceRegister(0b00000000)
print('\u521d\u59cb\u72b6\u6001:')
reg.show()

print('--- \u6253\u5f00\u7535\u6e90 ---')
reg.set_bit(7)
reg.show()

print('--- \u6253\u5f00\u7ea2\u8272\u548c\u84dd\u8272LED ---')
reg.set_bit(3)
reg.set_bit(1)
reg.show()

print('--- \u5f00\u542f\u95ea\u70c1\u6a21\u5f0f ---')
reg.set_bit(4)
reg.show()

print('--- \u5173\u95ed\u7ea2\u8272LED ---')
reg.clear_bit(3)
reg.show()

print('--- \u6d4b\u8bd5\u84dd\u8272LED\u72b6\u6001 ---')
reg.test_bit(1)

## 8. 实例：读取传感器状态

假设一个8位寄存器表示8个传感器的状态（1=触发，0=正常）：

In [ ]:
def read_sensors(sensor_byte):
    sensor_names = [
        '\u6e29\u5ea6\u4f20\u611f\u5668', '\u6e7f\u5ea6\u4f20\u611f\u5668', '\u5149\u7ebf\u4f20\u611f\u5668', '\u538b\u529b\u4f20\u611f\u5668',
        '\u70df\u96fe\u4f20\u611f\u5668', '\u8fd0\u52a8\u4f20\u611f\u5668', '\u95e8\u7a97\u4f20\u611f\u5668', '\u6c34\u6d78\u4f20\u611f\u5668'
    ]
    print(f'\u4f20\u611f\u5668\u72b6\u6001\u5b57\u8282: {format(sensor_byte, "08b")} (0x{sensor_byte:02X})\n')
    
    triggered = []
    for i in range(8):
        mask = 1 << i
        is_triggered = (sensor_byte & mask) != 0
        status = '\u26a0\ufe0f \u89e6\u53d1!' if is_triggered else '  \u6b63\u5e38'
        print(f'  Sensor {i} ({sensor_names[i]:>8}): {status}')
        if is_triggered:
            triggered.append(sensor_names[i])
    
    if triggered:
        print(f'\n\u8b66\u62a5: {len(triggered)} \u4e2a\u4f20\u611f\u5668\u89e6\u53d1 - {", ".join(triggered)}')
    else:
        print('\n\u6240\u6709\u4f20\u611f\u5668\u6b63\u5e38')

# Example: sensors 0, 2, 4 triggered
print('=== \u573a\u666f1: \u591a\u4e2a\u4f20\u611f\u5668\u89e6\u53d1 ===')
read_sensors(0b00010101)
print()
print('=== \u573a\u666f2: \u6240\u6709\u6b63\u5e38 ===')
read_sensors(0b00000000)
print()
print('=== \u573a\u666f3: \u7d27\u6025\u60c5\u51b5 ===')
read_sensors(0b11111111)

## 9. 交互式位操作工具

一个完整的位操作演示工具：

In [ ]:
class BitManipulator:
    def __init__(self, bits=8):
        self.bits = bits
        self.value = 0
    
    def show(self, label=''):
        fmt = f'0{self.bits}b'
        print(f'{label:>20}: {format(self.value, fmt)} = {self.value}')
    
    def demo_all_operations(self, a, b):
        fmt = f'0{self.bits}b'
        print(f'\n{"="*50}')
        print(f'\u4f4d\u64cd\u4f5c\u6f14\u793a: A = {a} ({format(a, fmt)}), B = {b} ({format(b, fmt)})')
        print(f'{"="*50}\n')
        
        operations = [
            ('A AND B', a & b, '\u53ea\u4fdd\u7559\u4e24\u8005\u90fd\u4e3a1\u7684\u4f4d'),
            ('A OR B', a | b, '\u4efb\u4e00\u4e3a1\u5219\u7ed3\u679c\u4e3a1'),
            ('A XOR B', a ^ b, '\u4e0d\u540c\u4e3a1\uff0c\u76f8\u540c\u4e3a0'),
            ('NOT A', (~a) & ((1 << self.bits) - 1), '\u6240\u6709\u4f4d\u53d6\u53cd'),
            ('A << 1', (a << 1) & ((1 << self.bits) - 1), f'\u5de6\u79fb1\u4f4d (\u00d72) = {a*2}'),
            ('A >> 1', a >> 1, f'\u53f3\u79fb1\u4f4d (\u00f72) = {a//2}'),
            ('A << 2', (a << 2) & ((1 << self.bits) - 1), f'\u5de6\u79fb2\u4f4d (\u00d74) = {a*4}'),
            ('A >> 2', a >> 2, f'\u53f3\u79fb2\u4f4d (\u00f74) = {a//4}'),
        ]
        
        for name, result, desc in operations:
            print(f'  {name:>10} = {format(result, fmt)} = {result:>3}  | {desc}')
        
        # XOR encryption demo
        print(f'\n--- XOR\u52a0\u5bc6\u6f14\u793a ---')
        key = b
        encrypted = a ^ key
        decrypted = encrypted ^ key
        print(f'  \u539f\u6587:     {format(a, fmt)} ({a})')
        print(f'  \u5bc6\u94a5:     {format(key, fmt)} ({key})')
        print(f'  \u52a0\u5bc6\u540e:   {format(encrypted, fmt)} ({encrypted})')
        print(f'  \u89e3\u5bc6\u540e:   {format(decrypted, fmt)} ({decrypted})')
        print(f'  \u539f\u6587==\u89e3\u5bc6? {a == decrypted} \u2714')

bm = BitManipulator(8)
bm.demo_all_operations(0b10110101, 0b00001111)
bm.demo_all_operations(0b11001010, 0b10101010)

## 10. 所有移位类型综合可视化

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(5, 1, figsize=(13, 12))
fig.patch.set_facecolor('#1a1a2e')

shift_demos = [
    ('Logical Left Shift (\u903b\u8f91\u5de6\u79fb)', '00101100', '01011000', '0\u586b\u5145\u53f3\u7aef, \u5de6\u7aef\u4e22\u5f03', '#e94560'),
    ('Logical Right Shift (\u903b\u8f91\u53f3\u79fb)', '00101100', '00010110', '0\u586b\u5145\u5de6\u7aef, \u53f3\u7aef\u4e22\u5f03', '#ffd700'),
    ('Arithmetic Right Shift (\u7b97\u672f\u53f3\u79fb)', '11110100', '11111010', '\u7b26\u53f7\u4f4d\u586b\u5145\u5de6\u7aef', '#00d2ff'),
    ('Cyclic Left Shift (\u5faa\u73af\u5de6\u79fb)', '10110011', '01100111', '\u5de6\u7aef\u79fb\u51fa\u7684\u4f4d\u8fdb\u5165\u53f3\u7aef', '#53d769'),
    ('Cyclic Right Shift (\u5faa\u73af\u53f3\u79fb)', '10110011', '11011001', '\u53f3\u7aef\u79fb\u51fa\u7684\u4f4d\u8fdb\u5165\u5de6\u7aef', '#ff9ff3'),
]

for idx, (title, before, after, note, color) in enumerate(shift_demos):
    ax = axes[idx]
    ax.set_facecolor('#16213e')
    ax.axis('off')
    ax.set_xlim(0, 14)
    ax.set_ylim(0, 2)
    
    ax.text(0.2, 1.3, title, fontsize=10, fontweight='bold', color=color)
    
    # Before bits
    for j, bit in enumerate(before):
        x = 0.5 + j * 1.0
        box = mpatches.FancyBboxPatch((x, 0.3), 0.7, 0.6,
            boxstyle='round,pad=0.03', facecolor='#0f3460', edgecolor='#555', linewidth=1)
        ax.add_patch(box)
        ax.text(x+0.35, 0.6, bit, fontsize=11, color='white', ha='center', fontweight='bold')
    
    ax.text(8.8, 0.5, '\u2192', fontsize=16, color=color, ha='center', fontweight='bold')
    
    # After bits
    for j, bit in enumerate(after):
        x = 9.3 + j * 1.0
        # Highlight changed bits
        changed = (before[j] != bit)
        ec = color if changed else '#555'
        lw_v = 2 if changed else 1
        box = mpatches.FancyBboxPatch((x, 0.3), 0.7, 0.6,
            boxstyle='round,pad=0.03', facecolor='#0f3460', edgecolor=ec, linewidth=lw_v)
        ax.add_patch(box)
        ax.text(x+0.35, 0.6, bit, fontsize=11, color=color if changed else 'white',
            ha='center', fontweight='bold')
    
    ax.text(13.5, 1.3, note, fontsize=8, color='#a0d2db', ha='right')

plt.suptitle('All Types of Binary Shifts', fontsize=14, fontweight='bold', color='white')
plt.tight_layout()
plt.show()

## 11. 练习题

### 题目1
对二进制数 `01101010` 执行以下操作，写出结果：
- (a) 逻辑左移 2 位
- (b) 逻辑右移 1 位
- (c) AND 掩码 `11110000`
- (d) OR 掩码 `00001111`
- (e) XOR 掩码 `11111111`

### 题目2
解释为什么左移一位等于乘以2，右移一位等于除以2。用十进制类比解释。

### 题目3
一个设备控制寄存器的值为 `10100110`。
- (a) 如何只将 Bit 3 设置为 1，不改变其他位？写出所需的掩码和操作。
- (b) 如何只将 Bit 5 清除为 0，不改变其他位？
- (c) 如何测试 Bit 7 是否为 1？

### 题目4
解释算术右移和逻辑右移的区别。为什么处理有符号数时必须使用算术右移？

### 题目5
解释 XOR 加密的原理。为什么 `A XOR K XOR K = A`？

In [ ]:
# \u7ec3\u4e60\u9898\u7b54\u6848\u68c0\u67e5
val = 0b01101010
print('\u9898\u76ee1 \u7b54\u6848:')
print(f'\u539f\u503c:     {format(val, "08b")} ({val})')
print(f'(a) <<2:    {format((val << 2) & 0xFF, "08b")} ({(val << 2) & 0xFF})')
print(f'(b) >>1:    {format(val >> 1, "08b")} ({val >> 1})')
print(f'(c) AND F0: {format(val & 0xF0, "08b")} ({val & 0xF0})')
print(f'(d) OR  0F: {format(val | 0x0F, "08b")} ({val | 0x0F})')
print(f'(e) XOR FF: {format(val ^ 0xFF, "08b")} ({val ^ 0xFF})')

print()
print('\u9898\u76ee3 \u7b54\u6848:')
reg_val = 0b10100110
print(f'\u539f\u59cb\u503c: {format(reg_val, "08b")}')
print(f'(a) \u8bbe\u7f6eBit3: OR  {format(1<<3, "08b")} -> {format(reg_val | (1<<3), "08b")}')
print(f'(b) \u6e05\u9664Bit5: AND {format((~(1<<5)) & 0xFF, "08b")} -> {format(reg_val & ~(1<<5) & 0xFF, "08b")}')
print(f'(c) \u6d4b\u8bd5Bit7: AND {format(1<<7, "08b")} -> {"\u4e3a1" if (reg_val & (1<<7)) else "\u4e3a0"}')

## 12. Python 位操作速查表

| 操作 | Python语法 | 例子 | 结果 |
|------|-----------|------|------|
| AND | `a & b` | `0b1010 & 0b1100` | `0b1000` |
| OR | `a \| b` | `0b1010 \| 0b1100` | `0b1110` |
| XOR | `a ^ b` | `0b1010 ^ 0b1100` | `0b0110` |
| NOT | `~a` | `~0b1010` | `-0b1011` |
| Left Shift | `a << n` | `0b0011 << 2` | `0b1100` |
| Right Shift | `a >> n` | `0b1100 >> 2` | `0b0011` |

> **提示**：Python中可以直接用 `0b` 前缀写二进制数，用 `bin()` 查看二进制表示。

In [ ]:
# \u5feb\u901f\u5b9e\u9a8c\u5de5\u5177
print('\u8bd5\u8bd5\u770b\uff01\u4fee\u6539\u4e0b\u9762\u7684\u503c\u8fdb\u884c\u5b9e\u9a8c:\n')

a = 0b10110101
b = 0b11001010

print(f'a = {a:08b} ({a})')
print(f'b = {b:08b} ({b})')
print(f'a & b  = {a & b:08b} ({a & b})')
print(f'a | b  = {a | b:08b} ({a | b})')
print(f'a ^ b  = {a ^ b:08b} ({a ^ b})')
print(f'~a     = {(~a) & 0xFF:08b} ({(~a) & 0xFF})')
print(f'a << 1 = {(a << 1) & 0xFF:08b} ({(a << 1) & 0xFF})')
print(f'a >> 1 = {a >> 1:08b} ({a >> 1})')
print()
print('\u6709\u8da3\u7684\u5e94\u7528: \u4e0d\u7528\u4e34\u65f6\u53d8\u91cf\u4ea4\u6362\u4e24\u4e2a\u6570')
x, y = 42, 73
print(f'\u4ea4\u6362\u524d: x={x}, y={y}')
x = x ^ y
y = x ^ y
x = x ^ y
print(f'\u4ea4\u6362\u540e: x={x}, y={y}')

## 13. 本章总结

### 移位运算
- **逻辑左移**：所有位左移，右边填0，相当于乘2
- **逻辑右移**：所有位右移，左边填0，相当于无符号除2
- **算术右移**：左边填符号位，保持有符号数的正负
- **循环移位**：移出的位从另一端重新进入

### 位掩码
- **AND**：测试位、清除位
- **OR**：设置位
- **XOR**：翻转位、加密

### 实际应用
- 设备控制寄存器
- 传感器状态读取
- 快速乘除运算
- 数据加密

---

**恭喜你完成了第4章的全部内容！** 你现在理解了：
1. CPU的架构和各组件功能
2. 取指-执行周期的详细步骤
3. 汇编语言的指令和寻址方式
4. 位操作的原理和应用

这些知识构成了理解计算机工作原理的基础！